# Paper 03 · LeNet / Gradient-Based Document Recognition

**Citation:** Yann LeCun et al., “Gradient-Based Learning Applied to Document Recognition” (1998).

**Paper:** http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf

> **Scale gap:** We use scikit-learn's 8×8 digits instead of MNIST and train a tiny LeNet-style CNN on CPU.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. Why should the same detector be reused across image locations?
2. What parameter-count advantage does convolution provide?
3. What would a fair dense-network baseline look like?

## Central claim
Convolution, local receptive fields, shared weights, and subsampling provide useful inductive biases for image recognition.

## Load a no-download image dataset

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits=load_digits()
X=(digits.images/16.0).astype("float32")[:,None,:,:]
y=digits.target.astype("int64")
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
Xtr=torch.tensor(Xtr); Xte=torch.tensor(Xte); ytr=torch.tensor(ytr); yte=torch.tensor(yte)
plt.imshow(Xtr[0,0],cmap="gray"); plt.title(f"label={ytr[0].item()}"); plt.show()

## Partially completed LeNet-style model

In [ ]:
class TinyLeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,8,3,padding=1),nn.Tanh(),
            nn.AvgPool2d(2),
            nn.Conv2d(8,16,3,padding=1),nn.Tanh(),
            nn.AvgPool2d(2),
        )
        # TODO: calculate why the flattened size is 16*2*2.
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(16*2*2,32),nn.Tanh(),nn.Linear(32,10))
    def forward(self,x): return self.classifier(self.features(x))

class DenseBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(64,64),nn.Tanh(),nn.Linear(64,10))
    def forward(self,x): return self.net(x)

## Controlled training comparison

In [ ]:
def train(model,epochs=35):
    torch.manual_seed(0)
    opt=torch.optim.Adam(model.parameters(),lr=.01); loss_fn=nn.CrossEntropyLoss(); hist=[]
    for _ in range(epochs):
        opt.zero_grad(); loss=loss_fn(model(Xtr),ytr); loss.backward(); opt.step()
        with torch.no_grad(): acc=(model(Xte).argmax(1)==yte).float().mean().item()
        hist.append((loss.item(),acc))
    return np.array(hist)
models={"LeNet-style":TinyLeNet(),"Dense":DenseBaseline()}
results={}
for name,m in models.items():
    h=train(m); results[name]=(sum(p.numel() for p in m.parameters()),h[-1,1],h)
    print(name,"params",results[name][0],"test accuracy",results[name][1])
for name,(_,_,h) in results.items(): plt.plot(h[:,1],label=name)
plt.legend(); plt.ylabel("test accuracy"); plt.xlabel("epoch"); plt.show()

### Ablation
Replace average pooling with no pooling and adjust the dense input size. Compare parameter count, training speed, and accuracy.

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?